# H12 — 没有电脑也能配置网络

**学习入口：把下面 prompt 发给 VS Code Codex 侧边栏。** Notebook 也可自行逐格阅读。
C++ 在持续 ESP-IDF 工程编辑与构建；Notebook 运行仅显示示例/手动证据卡。
**状态：Alpha；固件构建与实机验收分开记录，见 ../VALIDATION.md。**

```text
开始 H12 — 没有电脑也能配置网络。
请先读取本工程 AGENTS.md、.academy/course.json 和 .academy/progress.md。
通过 course_root 找到 step2/course.json 和 step2/notebooks/H12-ble.ipynb。
读取 Notebook 的 cells/source 与 academy.step_id/mode；先读当前步骤相关格。
检查工程、环境和前置条件，从第一个未完成步骤开始；每轮只推进一个小步骤。
Guided 可合并指定示例并说明 diff；Modify/Debug/Build/Open 先让我尝试，按需逐级提示。
不要主动读取 solution.md 或其他答案。环境故障可以排查修复，解释后回到原步骤。
不要把挑战 Bug 自动修掉。以当前代码与真实证据验收，记录提示级别和下一步。
我已有一定 C++ 基础，Python 几乎零基础。现在开始。
```

不用 agent 时按步骤标题顺序操作；卡住后再逐级查看本课 hints.md。启动 prompt 的路径按工作区课程链接解析，不靠其他窗口或聊天记忆。

## 我们要解决什么问题？

把工具交给别人时，不应要求对方打开源码改 Wi-Fi。用手机 BLE 配网，复用已经建立的配置和网络职责。

前置：H11；支持 BLE 的板，手机与官方 app。

## 本次够用的知识

BLE 的广播帮助发现，GATT Service/Characteristic 表达交互；普通系统蓝牙设置不等于应用层 provisioning 客户端。Espressif provisioning 使用 protocomm，通用 BLE scanner 能看 GATT，但不能自动完成加密配网协议。

先预测、运行和记录，再补概念。每节开始保存现有工程差异，后续沿用同一工程。

## 可选：Notebook 示例与证据卡

使用课程 .venv kernel 运行下格；这只是显示工具，不需要学习 Python。侧边栏学习不依赖这些格，build/flash/monitor 都在 ESP-IDF Terminal 执行。

In [ ]:
from pathlib import Path
import sys
import json

course_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "step2/course.json").is_file():
        course_root = candidate
        break
    link = candidate / ".academy/course.json"
    if link.is_file():
        course_root = Path(json.loads(link.read_text(encoding="utf-8"))["course_root"])
        break
if course_root is None or not (course_root / "step2/course.json").is_file():
    raise RuntimeError("请从课程工作区运行，或先在学习工程 .academy/course.json 链接课程根目录")
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))
from tools.esp32_notebook import install, hardware_checklist
install(get_ipython())


## 核对无线能力与客户端 · Guided

经典 ESP32 支持 BLE；ESP32-S2 不支持 BLE。确认板型后选择 ESP-IDF BLE provisioning；若硬件无 BLE，用 SoftAP 作为不同路线，不伪造成功。
安装 Espressif 官方 provisioning 手机 app。记录已有 NVS 凭据与恢复方式；不为了广播自动清空 Wi-Fi 配置。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 真实 BLE 配网 · Guided

REQUIRES wifi_provisioning esp_wifi esp_event esp_netif nvs_flash freertos。配置 CONFIG_BT_ENABLED=y、CONFIG_BT_NIMBLE_ENABLED=y、经典芯片 BLE-only；参考官方例程的 sdkconfig.defaults。
示例替换 H07 的凭据初始化路径，不能再调用 wifi_link_start 初始化第二个 event loop。首次未 provisioned 才广播；配过网的设备走已配置路径。教学 PoP academy-demo 仅作本地试验，部署需每设备唯一值。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

示例来源：`step2/lessons/H12/examples/ble_provision.cpp`。目标：`main/main.cpp`。这是教学示例；合并后以工程源码为准，后续不要来回修改两份实现。依赖参考：`REQUIRES wifi_provisioning esp_wifi esp_event esp_netif nvs_flash freertos log`；SRCS 按本步骤目标文件登记。

In [ ]:
%%esp32_example --target main/main.cpp
#include "esp_event.h"
#include "esp_log.h"
#include "esp_netif.h"
#include "esp_wifi.h"
#include "nvs_flash.h"
#include "wifi_provisioning/manager.h"
#include "wifi_provisioning/scheme_ble.h"
#include "freertos/FreeRTOS.h"
#include "freertos/task.h"

static void event(void*, esp_event_base_t base, int32_t id, void* data) {
    if (base == WIFI_PROV_EVENT && id == WIFI_PROV_CRED_SUCCESS) {
        ESP_LOGI("provision", "credentials accepted (never log password)");
    } else if (base == WIFI_PROV_EVENT && id == WIFI_PROV_END) {
        wifi_prov_mgr_deinit();
    } else if (base == WIFI_EVENT && id == WIFI_EVENT_STA_START) {
        ESP_ERROR_CHECK(esp_wifi_connect());
    } else if (base == WIFI_EVENT && id == WIFI_EVENT_STA_DISCONNECTED) {
        ESP_LOGW("provision", "disconnected; add your H07 backoff policy here");
    } else if (base == IP_EVENT && id == IP_EVENT_STA_GOT_IP) {
        auto* ip = static_cast<ip_event_got_ip_t*>(data);
        ESP_LOGI("provision", "IP=" IPSTR, IP2STR(&ip->ip_info.ip));
    }
}
extern "C" void app_main() {
    // This demonstrates replacement of credential setup, not a second Wi-Fi stack.
    ESP_ERROR_CHECK(nvs_flash_init());
    ESP_ERROR_CHECK(esp_netif_init());
    ESP_ERROR_CHECK(esp_event_loop_create_default());
    esp_netif_create_default_wifi_sta();
    wifi_init_config_t wifi = WIFI_INIT_CONFIG_DEFAULT();
    ESP_ERROR_CHECK(esp_wifi_init(&wifi));
    ESP_ERROR_CHECK(
        esp_event_handler_register(WIFI_PROV_EVENT, ESP_EVENT_ANY_ID, event, nullptr));
    ESP_ERROR_CHECK(
        esp_event_handler_register(WIFI_EVENT, ESP_EVENT_ANY_ID, event, nullptr));
    ESP_ERROR_CHECK(
        esp_event_handler_register(IP_EVENT, IP_EVENT_STA_GOT_IP, event, nullptr));
    wifi_prov_mgr_config_t config{};
    config.scheme = wifi_prov_scheme_ble;
    config.scheme_event_handler = WIFI_PROV_SCHEME_BLE_EVENT_HANDLER_FREE_BTDM;
    ESP_ERROR_CHECK(wifi_prov_mgr_init(config));
    bool provisioned = false;
    ESP_ERROR_CHECK(wifi_prov_mgr_is_provisioned(&provisioned));
    if (!provisioned) {
        // Teaching PoP only: replace per device before deployment; do not claim production security.
        ESP_ERROR_CHECK(wifi_prov_mgr_start_provisioning(
            WIFI_PROV_SECURITY_1, "academy-demo", "ACADEMY_DESK", nullptr));
        ESP_LOGI("provision", "Use Espressif provisioning app: BLE, name=ACADEMY_DESK");
    } else {
        wifi_prov_mgr_deinit();
        ESP_ERROR_CHECK(esp_wifi_set_mode(WIFI_MODE_STA));
        ESP_ERROR_CHECK(esp_wifi_start());
    }
    while (true) {
        vTaskDelay(pdMS_TO_TICKS(1000));
    }
}


## 看见广播不等于配好网 · Modify

在 scanner 看 Service/Characteristic，在官方 app 选择 BLE 与 ACADEMY_DESK，输入教学 PoP 和本地 AP 凭据；日志不能打印密码。
分别记录被发现、建立会话、凭据接受、得到 IP 与 API 可访问。重启后不应自动重新进入配网。手机系统权限与普通蓝牙配对页失败分开诊断。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 有广告却没有授权 · Debug

在 app 使用错误 PoP，观察安全会话失败，不修改固件跳过安全机制。再尝试错误 Wi-Fi 密码，区分 BLE 会话成功与 AP 认证失败。
要重新测试配网时，设计明确用户操作（长按/串口命令）调用 wifi_prov_mgr_reset_provisioning；记录会影响的凭据，仅在用户选择后执行。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 有边界的配置入口 · Build

需求：配网模式有超时、物理进入方式、设备识别、失败重试和退出；错误输入不覆盖有效配置；配网完成后网络回连沿用 H07 退避。
若扩展自己的 GATT name/threshold Characteristic，完整验证长度/UTF-8/范围，访问授权单独设计；不把 demo PoP 当作生产级身份保证。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 交给另一位使用者 · Open

让另一个人只看使用说明完成发现/配网/访问，不提供开发环境。记录手机型号/OS/app 版本与实际失败。
H13 会解决交付后的更新；保存串口救援路径。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 验收与复盘

- [ ] 真实手机发现、会话、凭据接受、IP/API 分层记录
- [ ] 错误 PoP/密码不会被当成配网成功
- [ ] 重新配网入口、超时与凭据日志边界明确

分别记录 host、firmware build、flash、日志、实物与网络往返；未知保持未确认。手动卡片只是学习者记录，不自动测量，也不自动写入进度文件。

复盘：我选择了什么机制？哪个证据排除了错误假设？哪些内容还需要测量？把当前 Step、Hint level、代码版本、证据和下次起点写入 `.academy/progress.md`。

In [ ]:
hardware_checklist(
    'H12',
    [
        '真实手机发现、会话、凭据接受、IP/API 分层记录',
        '错误 PoP/密码不会被当成配网成功',
        '重新配网入口、超时与凭据日志边界明确',
    ],
)


## 遇到困难时

先完成一次尝试，再打开 `step2/lessons/H12/hints.md`，每次只读当前步骤的一层。完整参考与解题分析单独保存，默认路径不主动读取。

## 官方版本参考

- [api-reference/provisioning/wifi_provisioning.html](https://docs.espressif.com/projects/esp-idf/en/v5.5.1/esp32/api-reference/provisioning/wifi_provisioning.html)